# M1 Weak Positive 확장 학습

## tl;dr

- `Possible anomaly start`가 없어 strict positive에서 빠졌던 M1 `efd_possible=True` 후보를 `weak_positive`로 분리해 추가 검증한다.
- Event 20은 coverage 문제로 계속 제외하고, Event 34/69는 unknown fault label이라 audit에만 남긴다.
- 모델 고도화가 아니라 positive 확장만으로 최소 학습 신호가 안정되는지 확인하는 실험이다.

## Context & Methods

- 원본 `predist_dataset.zip`의 `manufacturer 1` metadata와 operational CSV만 사용한다.
- positive window는 전부 `Report date - 7 days`부터 `Report date`까지로 통일한다.
- normal은 기존 normal event 35건의 7일 window를 그대로 사용한다.
- feature는 M1 공통 10개 센서의 7개 통계 70개만 사용한다.
- 학습 입력에서는 날짜, event id, `substation_id`, coverage, label strength 같은 metadata를 제외한다.
- 모델은 `DummyClassifier`와 `LogisticRegression(class_weight="balanced")`만 비교한다.
- 검증은 `substation_id` 기준 group CV를 유지한다.

### Key Assumptions

- `Possible anomaly start`가 없는 positive는 strict가 아니라 `weak_positive`다.
- unknown fault label은 빠른 1차 확장 학습에서는 제외하고 audit 대상으로만 둔다.
- weak positive도 coverage가 0.95 미만이면 학습에서 제외한다.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings
import zipfile

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "05_데이터셋" / "PreDist"
ZIP_PATH = DATA_DIR / "predist_dataset.zip"
OUTPUT_DIR = ROOT / "07_데이터산출물"

AUDIT_PATH = OUTPUT_DIR / "m1_positive_expansion_audit.csv"
FEATURES_PATH = OUTPUT_DIR / "m1_positive_expansion_features.csv"
DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_positive_expansion_dataset_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_positive_expansion_cv_metrics.csv"
CV_PREDICTIONS_PATH = OUTPUT_DIR / "m1_positive_expansion_cv_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_positive_expansion_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_positive_expansion_feature_importance.csv"
REPORT_PATH = OUTPUT_DIR / "06_M1_positive_확장학습_보고서.md"

SOURCE_PREFIX = "manufacturer 1"
RANDOM_STATE = 42
POSITIVE_LABEL = "efd_possible"
COVERAGE_THRESHOLD = 0.95
EVENT20_ID = 20
EVENT67_ID = 67
UNKNOWN_REVIEW_IDS = {34, 69}

COMMON_SENSOR_COLUMNS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
SENSOR_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
FEATURE_COLUMNS = [f"{sensor}__{stat}" for sensor in COMMON_SENSOR_COLUMNS for stat in SENSOR_STATS]
THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
LABEL_TO_BINARY = {"normal": 0, POSITIVE_LABEL: 1}
BINARY_TO_LABEL = {0: "normal", 1: POSITIVE_LABEL}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Data

In [2]:
def read_zip_csv(zf: zipfile.ZipFile, relative_path: str) -> pd.DataFrame:
    with zf.open(f"{SOURCE_PREFIX}/{relative_path}") as file:
        return pd.read_csv(file, sep=";")


with zipfile.ZipFile(ZIP_PATH) as zf:
    faults = read_zip_csv(zf, "faults.csv")
    normal_events = read_zip_csv(zf, "normal_events.csv")
    disturbances = read_zip_csv(zf, "disturbances.csv")

for frame, columns in [
    (faults, ["Report date", "Possible anomaly start", "Possible anomaly end", "Training start", "Training end"]),
    (normal_events, ["Event start", "Event end", "Training start", "Training end"]),
    (disturbances, ["Event start"]),
]:
    for column in columns:
        frame[column] = pd.to_datetime(frame[column], errors="coerce")

faults["Event ID"] = faults["Event ID"].astype(int)
faults["substation ID"] = faults["substation ID"].astype(int)
normal_events["Event ID"] = normal_events["Event ID"].astype(int)
normal_events["substation ID"] = normal_events["substation ID"].astype(int)
disturbances["substation ID"] = disturbances["substation ID"].astype(int)

strict_mask = (
    faults["efd_possible"].eq(True)
    & faults["Possible anomaly start"].notna()
    & faults["Possible anomaly end"].notna()
    & faults["Training start"].notna()
    & faults["Training end"].notna()
    & faults["Report date"].notna()
)
strict_positive_ids = set(faults.loc[strict_mask, "Event ID"].astype(int))
weak_candidate_mask = faults["efd_possible"].eq(True) & ~faults["Event ID"].isin(strict_positive_ids)
weak_candidate_ids = set(faults.loc[weak_candidate_mask, "Event ID"].astype(int))

assert len(normal_events) == 35
assert len(strict_positive_ids) == 15
assert len(weak_candidate_ids) == 14
assert weak_candidate_ids == {5, 6, 13, 23, 24, 29, 32, 34, 37, 38, 45, 62, 65, 69}
assert UNKNOWN_REVIEW_IDS.issubset(weak_candidate_ids)

{
    "normal_events": len(normal_events),
    "strict_positive": len(strict_positive_ids),
    "weak_candidates": len(weak_candidate_ids),
}

{'normal_events': 35, 'strict_positive': 15, 'weak_candidates': 14}

In [3]:
def load_substation_frame(zf: zipfile.ZipFile, substation_id: int) -> pd.DataFrame:
    path = f"{SOURCE_PREFIX}/operational_data/substation_{int(substation_id)}.csv"
    with zf.open(path) as file:
        frame = pd.read_csv(file, sep=";", usecols=["timestamp", *COMMON_SENSOR_COLUMNS])
    frame["timestamp"] = pd.to_datetime(frame["timestamp"])
    for column in COMMON_SENSOR_COLUMNS:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


def get_substation_frame(cache: dict[int, pd.DataFrame], zf: zipfile.ZipFile, substation_id: int) -> pd.DataFrame:
    substation_id = int(substation_id)
    if substation_id not in cache:
        cache[substation_id] = load_substation_frame(zf, substation_id)
    return cache[substation_id]


def window_slice(frame: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    return frame[(frame["timestamp"] >= start) & (frame["timestamp"] < end)].copy()


def expected_row_count(start: pd.Timestamp, end: pd.Timestamp) -> float:
    return (end - start).total_seconds() / 600


def sensor_stats(window: pd.DataFrame) -> dict:
    output = {}
    sample_count = int(len(window))
    for column in COMMON_SENSOR_COLUMNS:
        series = window[column] if sample_count else pd.Series(dtype="float64")
        valid = series.dropna()
        output[f"{column}__mean"] = float(series.mean()) if sample_count else np.nan
        output[f"{column}__std"] = float(series.std()) if sample_count else np.nan
        output[f"{column}__min"] = float(series.min()) if sample_count else np.nan
        output[f"{column}__max"] = float(series.max()) if sample_count else np.nan
        output[f"{column}__median"] = float(series.median()) if sample_count else np.nan
        output[f"{column}__last_minus_first"] = float(valid.iloc[-1] - valid.iloc[0]) if len(valid) >= 2 else np.nan
        output[f"{column}__missing_rate"] = float(series.isna().mean()) if sample_count else np.nan
    return output


def disturbance_summary(substation_id: int, start: pd.Timestamp, end: pd.Timestamp) -> tuple[int, str]:
    selected = disturbances[
        (disturbances["substation ID"].eq(int(substation_id)))
        & (disturbances["Event start"] >= start)
        & (disturbances["Event start"] < end)
    ]
    type_summary = ",".join(
        f"{key}:{value}" for key, value in selected["type"].value_counts().sort_index().items()
    )
    return int(len(selected)), type_summary


def base_event_row(
    *,
    sample_id: str,
    label: str,
    label_strength: str,
    source_event_id: int,
    substation_id: int,
    source_file: str,
    window_start: pd.Timestamp,
    window_end: pd.Timestamp,
    window_policy: str,
    report_date: pd.Timestamp | None = None,
    possible_anomaly_start: pd.Timestamp | None = None,
    possible_anomaly_end: pd.Timestamp | None = None,
    training_start: pd.Timestamp | None = None,
    training_end: pd.Timestamp | None = None,
    fault_label: str | None = None,
    expansion_status: str = "accepted",
    exclusion_reason: str = "",
    event67_flag: bool = False,
) -> dict:
    window_days = (window_end - window_start).total_seconds() / 86400
    disturbance_count, disturbance_types = disturbance_summary(substation_id, window_start, window_end)
    return {
        "sample_id": sample_id,
        "manufacturer": "manufacturer_1",
        "label": label,
        "label_strength": label_strength,
        "source_file": source_file,
        "source_event_id": int(source_event_id),
        "substation_id": int(substation_id),
        "window_start": window_start,
        "window_end": window_end,
        "window_days": float(window_days),
        "window_policy": window_policy,
        "report_date": report_date,
        "possible_anomaly_start": possible_anomaly_start,
        "possible_anomaly_end": possible_anomaly_end,
        "training_start": training_start,
        "training_end": training_end,
        "fault_label": fault_label,
        "expansion_status": expansion_status,
        "exclusion_reason": exclusion_reason,
        "event67_flag": bool(event67_flag),
        "disturbance_count": disturbance_count,
        "disturbance_types": disturbance_types,
    }

In [4]:
def positive_status(row: pd.Series, coverage_rate: float) -> tuple[str, str, str]:
    event_id = int(row["Event ID"])
    fault_label = str(row["Fault label"])
    if event_id == EVENT20_ID:
        return "excluded", "event20_excluded", "low_coverage_event20"
    if event_id in UNKNOWN_REVIEW_IDS or fault_label.lower() == "unknown":
        reasons = ["unknown_fault_label"]
        if pd.isna(row["Training end"]):
            reasons.append("missing_training_end")
        return "excluded", "unknown_review", ";".join(reasons)
    if coverage_rate < COVERAGE_THRESHOLD:
        return "excluded", "low_coverage_excluded", "coverage_below_0.95"
    if event_id in strict_positive_ids:
        return "accepted", "strict_accepted", ""
    return "accepted", "weak_accepted", ""


feature_rows = []
positive_audit_rows = []
positive_all_feature_rows = []

with zipfile.ZipFile(ZIP_PATH) as zf:
    cache: dict[int, pd.DataFrame] = {}

    for _, row in normal_events.iterrows():
        substation_id = int(row["substation ID"])
        start = row["Event start"]
        end = row["Event end"]
        frame = get_substation_frame(cache, zf, substation_id)
        window = window_slice(frame, start, end)
        expected_count = expected_row_count(start, end)
        sample_count = int(len(window))
        event = base_event_row(
            sample_id=f"normal_{int(row['Event ID']):04d}",
            label="normal",
            label_strength="normal",
            source_event_id=int(row["Event ID"]),
            substation_id=substation_id,
            source_file=f"substation_{substation_id}.csv",
            window_start=start,
            window_end=end,
            window_policy="normal_event_start_to_end",
            training_start=row["Training start"],
            training_end=row["Training end"],
        )
        event.update(
            {
                "sample_count": sample_count,
                "expected_sample_count": expected_count,
                "coverage_rate": sample_count / expected_count if expected_count else np.nan,
            }
        )
        event.update(sensor_stats(window))
        feature_rows.append(event)

    positive_candidates = faults[faults["efd_possible"].eq(True)].copy()
    for _, row in positive_candidates.iterrows():
        event_id = int(row["Event ID"])
        substation_id = int(row["substation ID"])
        end = row["Report date"]
        start = end - pd.Timedelta(days=7)
        frame = get_substation_frame(cache, zf, substation_id)
        window = window_slice(frame, start, end)
        expected_count = expected_row_count(start, end)
        sample_count = int(len(window))
        coverage_rate = sample_count / expected_count if expected_count else np.nan
        label_strength = "strict_positive" if event_id in strict_positive_ids else "weak_positive"
        train_status, expansion_status, exclusion_reason = positive_status(row, coverage_rate)

        event = base_event_row(
            sample_id=f"{label_strength}_{event_id:04d}",
            label=POSITIVE_LABEL,
            label_strength=label_strength,
            source_event_id=event_id,
            substation_id=substation_id,
            source_file=f"substation_{substation_id}.csv",
            window_start=start,
            window_end=end,
            window_policy="report_date_minus_7d_to_report_date",
            report_date=row["Report date"],
            possible_anomaly_start=row["Possible anomaly start"],
            possible_anomaly_end=row["Possible anomaly end"],
            training_start=row["Training start"],
            training_end=row["Training end"],
            fault_label=row["Fault label"],
            expansion_status=expansion_status,
            exclusion_reason=exclusion_reason,
            event67_flag=event_id == EVENT67_ID,
        )
        event.update(
            {
                "sample_count": sample_count,
                "expected_sample_count": expected_count,
                "coverage_rate": coverage_rate,
                "train_status": train_status,
            }
        )
        event.update(sensor_stats(window))
        positive_all_feature_rows.append(event)

        audit_columns = [
            "source_event_id",
            "substation_id",
            "label_strength",
            "fault_label",
            "window_start",
            "window_end",
            "window_days",
            "sample_count",
            "expected_sample_count",
            "coverage_rate",
            "disturbance_count",
            "disturbance_types",
            "possible_anomaly_start",
            "possible_anomaly_end",
            "training_start",
            "training_end",
            "expansion_status",
            "exclusion_reason",
            "event67_flag",
        ]
        positive_audit_rows.append({column: event.get(column) for column in audit_columns})

        if train_status == "accepted":
            feature_rows.append(event)

features = pd.DataFrame(feature_rows)
positive_audit = pd.DataFrame(positive_audit_rows).sort_values("source_event_id").reset_index(drop=True)

for timestamp_column in [
    "window_start",
    "window_end",
    "report_date",
    "possible_anomaly_start",
    "possible_anomaly_end",
    "training_start",
    "training_end",
]:
    if timestamp_column in features.columns:
        features[timestamp_column] = pd.to_datetime(features[timestamp_column], errors="coerce")
    if timestamp_column in positive_audit.columns:
        positive_audit[timestamp_column] = pd.to_datetime(positive_audit[timestamp_column], errors="coerce")

assert set(features["manufacturer"]) == {"manufacturer_1"}
assert not features["source_event_id"].eq(EVENT20_ID).any()
assert not features["source_event_id"].isin(UNKNOWN_REVIEW_IDS).any()
assert positive_audit["source_event_id"].isin(UNKNOWN_REVIEW_IDS).sum() == 2
assert positive_audit["source_event_id"].eq(EVENT20_ID).sum() == 1
assert features[features["label_strength"].eq("weak_positive")]["coverage_rate"].ge(COVERAGE_THRESHOLD).all()
assert set(FEATURE_COLUMNS).issubset(features.columns)

positive_audit.to_csv(AUDIT_PATH, index=False, encoding="utf-8-sig")
features.to_csv(FEATURES_PATH, index=False, encoding="utf-8-sig")

features.groupby(["label", "label_strength"]).size().reset_index(name="rows")

,label,label_strength,rows
0,efd_possible,strict_positive,14
1,efd_possible,weak_positive,12
2,normal,normal,35


## Results

In [5]:
normal_pool = features[features["label_strength"].eq("normal")].copy()
strict_pool = features[features["label_strength"].eq("strict_positive")].copy()
weak_pool = features[features["label_strength"].eq("weak_positive")].copy()

datasets = {
    "strict_no_event20": pd.concat([normal_pool, strict_pool], ignore_index=True),
    "strict_no_event20_no_event67": pd.concat(
        [normal_pool, strict_pool[~strict_pool["source_event_id"].eq(EVENT67_ID)]],
        ignore_index=True,
    ),
    "expanded_weak_positive": pd.concat([normal_pool, strict_pool, weak_pool], ignore_index=True),
}

accepted_weak_count = int(len(weak_pool))
assert len(normal_pool) == 35
assert len(strict_pool) == 14
assert accepted_weak_count <= 12
assert len(datasets["strict_no_event20"]) == 49
assert (datasets["strict_no_event20"]["label"].eq(POSITIVE_LABEL)).sum() == 14
assert len(datasets["strict_no_event20_no_event67"]) == 48
assert (datasets["strict_no_event20_no_event67"]["label"].eq(POSITIVE_LABEL)).sum() == 13
assert (datasets["expanded_weak_positive"]["label"].eq(POSITIVE_LABEL)).sum() == 14 + accepted_weak_count

summary_rows = []
for dataset_id, dataset in datasets.items():
    positive_events = sorted(dataset.loc[dataset["label"].eq(POSITIVE_LABEL), "source_event_id"].astype(int))
    weak_events = sorted(dataset.loc[dataset["label_strength"].eq("weak_positive"), "source_event_id"].astype(int))
    summary_rows.append(
        {
            "dataset_id": dataset_id,
            "rows": int(len(dataset)),
            "normal_rows": int(dataset["label"].eq("normal").sum()),
            "positive_rows": int(dataset["label"].eq(POSITIVE_LABEL).sum()),
            "strict_positive_rows": int(dataset["label_strength"].eq("strict_positive").sum()),
            "weak_positive_rows": int(dataset["label_strength"].eq("weak_positive").sum()),
            "positive_event_ids": ",".join(map(str, positive_events)),
            "weak_event_ids": ",".join(map(str, weak_events)),
            "event20_included": bool(EVENT20_ID in positive_events),
            "event67_included": bool(EVENT67_ID in positive_events),
            "coverage_min": float(dataset["coverage_rate"].min()),
            "coverage_median": float(dataset["coverage_rate"].median()),
            "coverage_max": float(dataset["coverage_rate"].max()),
            "learning_feature_count": len(FEATURE_COLUMNS),
        }
    )

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(DATASET_SUMMARY_PATH, index=False, encoding="utf-8-sig")
dataset_summary

,dataset_id,rows,normal_rows,positive_rows,strict_positive_rows,weak_positive_rows,positive_event_ids,weak_event_ids,event20_included,event67_included,coverage_min,coverage_median,coverage_max,learning_feature_count
0,strict_no_event20,49,35,14,14,0,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",,False,True,0.99504,1.0,1.0,70
1,strict_no_event20_no_event67,48,35,13,13,0,"1,3,7,10,15,40,44,47,49,52,53,57,64",,False,False,0.99504,1.0,1.0,70
2,expanded_weak_positive,61,35,26,14,12,"1,3,5,6,7,10,13,15,23,24,29,32,37,38,40,44,45,...","5,6,13,23,24,29,32,37,38,45,62,65",False,True,0.99504,1.0,1.0,70


In [6]:
def make_models() -> dict[str, Pipeline]:
    return {
        "dummy_most_frequent": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "logistic_balanced": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
        return np.zeros(len(X), dtype=float)
    return model.predict(X).astype(float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if y_score is not None and len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_score))
    else:
        roc_auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_and_evaluate(dataset_id: str, dataset: pd.DataFrame):
    dataset = dataset.copy().reset_index(drop=True)
    dataset["dataset_id"] = dataset_id
    X = dataset[FEATURE_COLUMNS].copy()
    y = dataset["label"].map(LABEL_TO_BINARY).astype(int).to_numpy()
    groups = dataset["substation_id"].astype(str).to_numpy()

    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    metric_rows = []
    prediction_rows = []
    importance_rows = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
        train_groups = sorted(set(groups[train_index]))
        test_groups = sorted(set(groups[test_index]))
        assert set(train_groups).isdisjoint(test_groups)
        assert len(np.unique(y_train)) == 2

        for model_name, model in make_models().items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).astype(int)
            y_score = positive_scores(model, X_test)
            row = metric_row(y_test, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "model": model_name,
                    "fold": fold,
                    "train_rows": int(len(train_index)),
                    "test_rows": int(len(test_index)),
                    "train_groups": "|".join(train_groups),
                    "test_groups": "|".join(test_groups),
                    "test_normal": int((y_test == 0).sum()),
                    "test_efd_possible": int((y_test == 1).sum()),
                }
            )
            metric_rows.append(row)

            prediction_meta = dataset.iloc[test_index][
                [
                    "dataset_id",
                    "sample_id",
                    "label",
                    "label_strength",
                    "source_event_id",
                    "substation_id",
                    "window_start",
                    "window_end",
                    "window_days",
                    "coverage_rate",
                    "sample_count",
                    "disturbance_count",
                ]
            ].copy()
            prediction_meta["fold"] = fold
            prediction_meta["model"] = model_name
            prediction_meta["actual_binary"] = y_test
            prediction_meta["predicted_binary"] = y_pred
            prediction_meta["predicted_label"] = [BINARY_TO_LABEL[int(value)] for value in y_pred]
            prediction_meta["positive_score"] = y_score
            prediction_meta["is_correct"] = prediction_meta["actual_binary"].eq(
                prediction_meta["predicted_binary"]
            )
            prediction_rows.extend(prediction_meta.to_dict("records"))

            if model_name == "logistic_balanced":
                coefficients = model.named_steps["model"].coef_[0]
                for feature, coefficient in zip(FEATURE_COLUMNS, coefficients):
                    sensor, stat = feature.split("__", 1)
                    importance_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "fold": fold,
                            "feature": feature,
                            "sensor": sensor,
                            "stat": stat,
                            "coefficient": float(coefficient),
                            "abs_coefficient": float(abs(coefficient)),
                        }
                    )

    return metric_rows, prediction_rows, importance_rows


all_metric_rows = []
all_prediction_rows = []
all_importance_rows = []
for dataset_id, dataset in datasets.items():
    metric_rows, prediction_rows, importance_rows = train_and_evaluate(dataset_id, dataset)
    all_metric_rows.extend(metric_rows)
    all_prediction_rows.extend(prediction_rows)
    all_importance_rows.extend(importance_rows)

cv_metrics = pd.DataFrame(all_metric_rows)
cv_predictions = pd.DataFrame(all_prediction_rows)
fold_importance = pd.DataFrame(all_importance_rows)

cv_metrics.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions.to_csv(CV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

cv_metrics.groupby(["dataset_id", "model"]).agg(
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    fp_sum=("fp", "sum"),
    fn_sum=("fn", "sum"),
).reset_index()

,dataset_id,model,balanced_accuracy_mean,precision_mean,recall_mean,f1_mean,fp_sum,fn_sum
0,expanded_weak_positive,dummy_most_frequent,0.500000,0.000000,0.000000,0.000000,0,26
1,expanded_weak_positive,logistic_balanced,0.569524,0.575238,0.553333,0.489524,15,12
2,strict_no_event20,dummy_most_frequent,0.500000,0.000000,0.000000,0.000000,0,14
3,strict_no_event20,logistic_balanced,0.573810,0.300000,0.500000,0.371429,12,7
4,strict_no_event20_no_event67,dummy_most_frequent,0.500000,0.000000,0.000000,0.000000,0,13
5,strict_no_event20_no_event67,logistic_balanced,0.582143,0.333333,0.533333,0.402381,13,6


In [7]:
logistic_predictions = cv_predictions[cv_predictions["model"].eq("logistic_balanced")].copy()
threshold_rows = []
for dataset_id, dataset_predictions in logistic_predictions.groupby("dataset_id"):
    y_true = dataset_predictions["actual_binary"].to_numpy(dtype=int)
    y_score = dataset_predictions["positive_score"].to_numpy(dtype=float)
    normal_count = int((y_true == 0).sum())
    positive_count = int((y_true == 1).sum())
    for threshold in THRESHOLDS:
        y_pred = (y_score >= threshold).astype(int)
        row = metric_row(y_true, y_pred, y_score)
        row.update(
            {
                "dataset_id": dataset_id,
                "model": "logistic_balanced",
                "threshold": float(threshold),
                "rows": int(len(y_true)),
                "normal_rows": normal_count,
                "positive_rows": positive_count,
                "false_positive_rate": row["fp"] / normal_count if normal_count else math.nan,
                "false_negative_rate": row["fn"] / positive_count if positive_count else math.nan,
            }
        )
        threshold_rows.append(row)

threshold_metrics = pd.DataFrame(threshold_rows)
threshold_metrics.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")
threshold_metrics

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,dataset_id,model,threshold,rows,normal_rows,positive_rows,false_positive_rate,false_negative_rate
0,0.508197,0.531868,0.450000,0.692308,0.545455,0.551648,13,22,8,18,expanded_weak_positive,logistic_balanced,0.3,61,35,26,0.628571,0.307692
1,0.524590,0.536264,0.457143,0.615385,0.524590,0.551648,16,19,10,16,expanded_weak_positive,logistic_balanced,0.4,61,35,26,0.542857,0.384615
2,0.557377,0.554945,0.482759,0.538462,0.509091,0.551648,20,15,12,14,expanded_weak_positive,logistic_balanced,0.5,61,35,26,0.428571,0.461538
3,0.573770,0.554396,0.500000,0.423077,0.458333,0.551648,24,11,15,11,expanded_weak_positive,logistic_balanced,0.6,61,35,26,0.314286,0.576923
4,0.510204,0.550000,0.321429,0.642857,0.428571,0.604082,16,19,5,9,strict_no_event20,logistic_balanced,0.3,49,35,14,0.542857,0.357143
5,0.551020,0.535714,0.318182,0.500000,0.388889,0.604082,20,15,7,7,strict_no_event20,logistic_balanced,0.4,49,35,14,0.428571,0.500000
6,0.612245,0.578571,0.368421,0.500000,0.424242,0.604082,23,12,7,7,strict_no_event20,logistic_balanced,0.5,49,35,14,0.342857,0.500000
7,0.673469,0.621429,0.437500,0.500000,0.466667,0.604082,26,9,7,7,strict_no_event20,logistic_balanced,0.6,49,35,14,0.257143,0.500000
8,0.562500,0.554945,0.318182,0.538462,0.400000,0.648352,20,15,6,7,strict_no_event20_no_event67,logistic_balanced,0.3,48,35,13,0.428571,0.461538
9,0.604167,0.583516,0.350000,0.538462,0.424242,0.648352,22,13,6,7,strict_no_event20_no_event67,logistic_balanced,0.4,48,35,13,0.371429,0.461538


In [8]:
feature_importance = fold_importance.groupby(
    ["dataset_id", "feature", "sensor", "stat"], as_index=False
).agg(
    coefficient_mean=("coefficient", "mean"),
    coefficient_std=("coefficient", "std"),
    abs_coefficient_mean=("abs_coefficient", "mean"),
    abs_coefficient_std=("abs_coefficient", "std"),
)
feature_importance["rank_abs_coefficient"] = feature_importance.groupby("dataset_id")[
    "abs_coefficient_mean"
].rank(method="first", ascending=False).astype(int)
feature_importance = feature_importance.sort_values(["dataset_id", "rank_abs_coefficient"])
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

feature_importance.groupby("dataset_id").head(8)

,dataset_id,feature,sensor,stat,coefficient_mean,coefficient_std,abs_coefficient_mean,abs_coefficient_std,rank_abs_coefficient
42,expanded_weak_positive,p_net_return_temperature__last_minus_first,p_net_return_temperature,last_minus_first,-0.772716,0.268789,0.772716,0.268789,1
11,expanded_weak_positive,p_hc1_return_temperature__min,p_hc1_return_temperature,min,-0.702778,0.229697,0.702778,0.229697,2
67,expanded_weak_positive,s_hc1_supply_temperature_setpoint__min,s_hc1_supply_temperature_setpoint,min,0.675590,0.309101,0.675590,0.309101,3
7,expanded_weak_positive,p_hc1_return_temperature__last_minus_first,p_hc1_return_temperature,last_minus_first,-0.609637,0.134651,0.609637,0.134651,4
1,expanded_weak_positive,outdoor_temperature__max,outdoor_temperature,max,0.570550,0.221318,0.570550,0.221318,5
60,expanded_weak_positive,s_hc1_supply_temperature__min,s_hc1_supply_temperature,min,-0.554100,0.134006,0.554100,0.134006,6
46,expanded_weak_positive,p_net_return_temperature__min,p_net_return_temperature,min,-0.476513,0.484825,0.523638,0.419691,7
53,expanded_weak_positive,p_net_supply_temperature__min,p_net_supply_temperature,min,-0.502347,0.177509,0.502347,0.177509,8
112,strict_no_event20,p_net_return_temperature__last_minus_first,p_net_return_temperature,last_minus_first,-0.513669,0.344077,0.513669,0.344077,1
70,strict_no_event20,outdoor_temperature__last_minus_first,outdoor_temperature,last_minus_first,0.498511,0.279168,0.498511,0.279168,2


## Takeaways

In [9]:
def markdown_table(frame: pd.DataFrame, columns: list[str] | None = None) -> str:
    if columns is not None:
        frame = frame[columns].copy()
    formatted = frame.copy()
    for column in formatted.columns:
        formatted[column] = formatted[column].map(
            lambda value: ""
            if pd.isna(value)
            else f"{value:.4f}"
            if isinstance(value, float)
            else str(value)
        )
    header = "| " + " | ".join(formatted.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(formatted.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in formatted.astype(str).to_numpy()]
    return "\n".join([header, sep, *rows])


cv_summary = cv_metrics.groupby(["dataset_id", "model"], as_index=False).agg(
    accuracy_mean=("accuracy", "mean"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    fp_sum=("fp", "sum"),
    fn_sum=("fn", "sum"),
)
logistic_cv_summary = cv_summary[cv_summary["model"].eq("logistic_balanced")].copy()
dummy_cv_summary = cv_summary[cv_summary["model"].eq("dummy_most_frequent")][
    ["dataset_id", "balanced_accuracy_mean", "recall_mean", "f1_mean"]
].rename(
    columns={
        "balanced_accuracy_mean": "dummy_balanced_accuracy_mean",
        "recall_mean": "dummy_recall_mean",
        "f1_mean": "dummy_f1_mean",
    }
)
decision_summary = logistic_cv_summary.merge(dummy_cv_summary, on="dataset_id", how="left")
decision_summary["balanced_accuracy_lift_vs_dummy"] = (
    decision_summary["balanced_accuracy_mean"] - decision_summary["dummy_balanced_accuracy_mean"]
)
decision_summary["recall_lift_vs_dummy"] = (
    decision_summary["recall_mean"] - decision_summary["dummy_recall_mean"]
)

threshold_05 = threshold_metrics[threshold_metrics["threshold"].eq(0.5)].set_index("dataset_id")
decision_by_dataset = decision_summary.set_index("dataset_id")
strict_lift = float(decision_by_dataset.loc["strict_no_event20", "balanced_accuracy_lift_vs_dummy"])
expanded_lift = float(decision_by_dataset.loc["expanded_weak_positive", "balanced_accuracy_lift_vs_dummy"])
strict_05 = threshold_05.loc["strict_no_event20"]
expanded_05 = threshold_05.loc["expanded_weak_positive"]

top_strict = set(
    feature_importance[
        (feature_importance["dataset_id"].eq("strict_no_event20"))
        & (feature_importance["rank_abs_coefficient"] <= 10)
    ]["feature"]
)
top_expanded = set(
    feature_importance[
        (feature_importance["dataset_id"].eq("expanded_weak_positive"))
        & (feature_importance["rank_abs_coefficient"] <= 10)
    ]["feature"]
)
top_feature_overlap = len(top_strict & top_expanded)
top_feature_jaccard = top_feature_overlap / len(top_strict | top_expanded) if (top_strict | top_expanded) else 0

if (
    expanded_lift >= strict_lift + 0.02
    and expanded_05["recall"] >= strict_05["recall"] - 0.05
    and expanded_05["false_positive_rate"] <= strict_05["false_positive_rate"]
    and expanded_05["false_positive_rate"] <= 0.35
    and top_feature_overlap >= 4
):
    conclusion = "`expanded_weak_positive`를 다음 기준 후보로 채택"
    conclusion_reason = (
        "weak positive 추가 후 Dummy 대비 개선폭이 커지고, recall과 false positive가 strict 기준보다 나쁘지 않았다."
    )
else:
    conclusion = "positive 확장만으로는 부족, 라벨 재검토/feature 확장 필요"
    conclusion_reason = (
        "weak positive 추가가 안정적인 개선으로 이어지는지, 또는 false positive를 충분히 낮추는지 확인되지 않았다."
    )

audit_summary = positive_audit.groupby(["label_strength", "expansion_status"], as_index=False).agg(
    rows=("source_event_id", "count"),
    coverage_min=("coverage_rate", "min"),
    coverage_median=("coverage_rate", "median"),
)
top_features = feature_importance[feature_importance["rank_abs_coefficient"] <= 8].copy()

report = f'''# M1 positive 확장학습 보고서

## 개요

strict positive만으로는 positive 샘플이 너무 작아, `efd_possible=True`지만 `Possible anomaly start`가 없는 M1 후보를 `weak_positive`로 분리해 추가 검증했다. 목적은 모델 고도화가 아니라 positive 확장만으로 최소 학습 신호가 안정되는지 확인하는 것이다.

## 결론

{conclusion}

- 판단 이유: {conclusion_reason}
- accepted weak positive: {accepted_weak_count}건
- unknown fault label Event 34/69는 학습에서 제외하고 audit에만 남겼다.
- Event 20은 계속 학습에서 제외했다.
- top-10 feature overlap(strict vs expanded): {top_feature_overlap}개, Jaccard {top_feature_jaccard:.4f}

## Positive audit 요약

{markdown_table(audit_summary, ["label_strength", "expansion_status", "rows", "coverage_min", "coverage_median"])}

## Dataset 요약

{markdown_table(dataset_summary, ["dataset_id", "rows", "normal_rows", "positive_rows", "strict_positive_rows", "weak_positive_rows", "event20_included", "event67_included", "coverage_min", "learning_feature_count"])}

## Group CV 요약

{markdown_table(decision_summary, ["dataset_id", "model", "balanced_accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean", "fp_sum", "fn_sum", "balanced_accuracy_lift_vs_dummy"])}

## Threshold 0.3~0.6 비교

{markdown_table(threshold_metrics, ["dataset_id", "threshold", "balanced_accuracy", "precision", "recall", "f1", "fp", "fn", "false_positive_rate", "false_negative_rate"])}

## 상위 feature 후보

{markdown_table(top_features, ["dataset_id", "rank_abs_coefficient", "feature", "coefficient_mean", "abs_coefficient_mean"])}

## 해석

- weak positive는 strict positive와 섞지 않고 `label_strength`로 구분했다.
- unknown fault label은 빠른 확장 학습에서 제외했기 때문에 후속 검토가 필요하다.
- feature 중요도는 샘플이 적은 상태의 반복 검증 후보이며 확정 근거가 아니다.
- 다음 단계는 결과가 안정적이면 weak positive 정책을 문서화하고, 불안정하면 라벨 재검토 또는 feature 확장으로 넘어가는 것이다.

## 생성 파일

- `07_데이터산출물/m1_positive_expansion_audit.csv`
- `07_데이터산출물/m1_positive_expansion_features.csv`
- `07_데이터산출물/m1_positive_expansion_dataset_summary.csv`
- `07_데이터산출물/m1_positive_expansion_cv_metrics.csv`
- `07_데이터산출물/m1_positive_expansion_cv_predictions.csv`
- `07_데이터산출물/m1_positive_expansion_threshold_metrics.csv`
- `07_데이터산출물/m1_positive_expansion_feature_importance.csv`

## 한계와 주의점

- weak positive는 anomaly start가 없어 strict positive보다 라벨 신뢰도가 낮다.
- group CV는 누수를 줄이지만, positive 수가 여전히 작아 fold별 변동이 크다.
- 모델 파일 저장과 운영 threshold 확정은 이번 단계에 포함하지 않았다.
'''

report = "\n".join(line.strip() for line in report.splitlines())
REPORT_PATH.write_text(report, encoding="utf-8")
print(conclusion)
print(REPORT_PATH)

positive 확장만으로는 부족, 라벨 재검토/feature 확장 필요
C:\3rd_Project\HeatGridAgent\07_데이터산출물\06_M1_positive_확장학습_보고서.md


In [10]:
output_paths = [
    AUDIT_PATH,
    FEATURES_PATH,
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    REPORT_PATH,
]
for path in output_paths:
    assert path.exists(), path

assert len(positive_audit) == 29
assert positive_audit["source_event_id"].isin([34, 69]).sum() == 2
assert positive_audit.loc[
    positive_audit["source_event_id"].isin([34, 69]), "expansion_status"
].eq("unknown_review").all()
assert not features["source_event_id"].eq(EVENT20_ID).any()
assert not features["source_event_id"].isin([34, 69]).any()
assert features[features["label_strength"].eq("weak_positive")]["coverage_rate"].ge(COVERAGE_THRESHOLD).all()

summary_index = dataset_summary.set_index("dataset_id")
assert int(summary_index.loc["strict_no_event20", "positive_rows"]) == 14
assert int(summary_index.loc["strict_no_event20_no_event67", "positive_rows"]) == 13
assert int(summary_index.loc["expanded_weak_positive", "positive_rows"]) == 14 + accepted_weak_count
assert not dataset_summary["event20_included"].any()

assert len(cv_metrics) == 30
assert len(threshold_metrics) == 12
assert len(feature_importance) == len(datasets) * len(FEATURE_COLUMNS)
assert set(feature_importance["sensor"].unique()) == set(COMMON_SENSOR_COLUMNS)
assert set(feature_importance["stat"].unique()) == set(SENSOR_STATS)

overlap_count = 0
for _, row in cv_metrics.iterrows():
    train_groups = set(str(row["train_groups"]).split("|"))
    test_groups = set(str(row["test_groups"]).split("|"))
    if train_groups & test_groups:
        overlap_count += 1
assert overlap_count == 0

{
    "accepted_weak_count": accepted_weak_count,
    "feature_rows": len(features),
    "metric_rows": len(cv_metrics),
    "prediction_rows": len(cv_predictions),
    "threshold_rows": len(threshold_metrics),
    "feature_importance_rows": len(feature_importance),
}

{'accepted_weak_count': 12,
 'feature_rows': 61,
 'metric_rows': 30,
 'prediction_rows': 316,
 'threshold_rows': 12,
 'feature_importance_rows': 210}